In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt
# import cv2
from tqdm import tqdm


import ast
import re


# capture/ videos index_df saved

In [25]:
capture_path = '/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/'

index_df = pd.read_csv(capture_path + 'index.csv')
index_df.shape

(50, 3)

In [26]:
index_df['session'] = ['plug_1']*5 + ['plug_2']*5 + ['magnet_1']*5 + ['magnet_2']*5 + ['books_1']*5 + ['books_2']*5 + ['lady_1']*5 + ['lady_2']*5 + ['paper_1']*5 + ['paper_2']*5

In [27]:
index_df

,id,template,placeholders,session
0,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1
1,260922-134301,Plugging [something] into [something],"plug, electrical extension",plug_1
2,260922-134333,Plugging [something] into [something],"plug, electrical extension",plug_1
3,260922-134419,Plugging [something] into [something],"plug, electrical extension",plug_1
4,260922-134433,Plugging [something] into [something],"plug, electrical extension",plug_1
5,260922-141236,Putting [something] next to [something],"water bottle, board marker",plug_2
6,260922-141249,Putting [something] on a surface,cup,plug_2
7,260922-141308,Moving [something] up,wallet,plug_2
8,260922-141343,Putting [something] onto [something],"pen, plate",plug_2
9,260922-141431,Pushing [something] with [something],"phone, water bottle",plug_2


In [28]:
# index_df expand by creating duplicate rows for each row by adding a new column 'cam': ['cam0','cam1','cam3']
index_df = index_df.loc[index_df.index.repeat(3)].reset_index(drop=True)
index_df['cam'] = ['cam0','cam1','cam2']*50
index_df.shape

(150, 5)

In [29]:
index_df.head(15)

,id,template,placeholders,session,cam
0,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam0
1,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1
2,260922-134214,Plugging [something] into [something],"plug, electrical extension",plug_1,cam2
3,260922-134301,Plugging [something] into [something],"plug, electrical extension",plug_1,cam0
4,260922-134301,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1
5,260922-134301,Plugging [something] into [something],"plug, electrical extension",plug_1,cam2
6,260922-134333,Plugging [something] into [something],"plug, electrical extension",plug_1,cam0
7,260922-134333,Plugging [something] into [something],"plug, electrical extension",plug_1,cam1
8,260922-134333,Plugging [something] into [something],"plug, electrical extension",plug_1,cam2
9,260922-134419,Plugging [something] into [something],"plug, electrical extension",plug_1,cam0


In [30]:
index_df.to_csv('./data/index_df.csv', index=False)

# ssv2 val set filtered

In [13]:
ssv2_val_label_file_path = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/public/ssv2/labels/validation.json"

with open(ssv2_val_label_file_path, 'r') as f:
    val_raw = json.load(f)

df = pd.DataFrame(val_raw)
df['split']='val'
print(df.shape)
df.head()

(24777, 5)


,id,label,template,placeholders,split
0,74225,spinning cube that quickly stops spinning,Spinning [something] that quickly stops spinning,[cube],val
1,116154,showing clay box on top of wallet,Showing [something] on top of [something],"[clay box, wallet]",val
2,198186,wiping words off of a paper,Wiping [something] off of [something],"[words, a paper]",val
3,137878,pushing scissors so that it falls off the table,Pushing [something] so that it falls off the t...,[scissors],val
4,151151,turning the camera left while filming wall mou...,Turning the camera left while filming [something],[wall mounted fan],val


In [17]:
df_final = df[df.template.isin(index_df.template.unique())]
df_final.shape      

(2543, 5)

In [18]:
df_final.template.value_counts()

template
Tearing [something] into two pieces             530
Plugging [something] into [something]           465
Moving [something] up                           359
Putting [something] on a surface                300
Putting [something] next to [something]         203
Pushing [something] with [something]            182
Piling [something] up                           161
Attaching [something] to [something]            152
Putting [something] onto [something]            139
Showing a photo of [something] to the camera     52
Name: count, dtype: int64

In [20]:
df_final.to_csv('./data/ssv2_val_filtered.csv', index=False)

In [ ]:
python scripts/vjepa_on_ssv2.py \
        --video_dir /user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/public/ssv2/clips/ \
        --batch_size 32 \
        --num_workers 25 \
        --fp16